# Business Entity Resolution: Amazon ML Challenge 2026
**Goal:** maximise macro F0.5 on the hidden test set. For each Source-1 entity, list all matching Source-2/Source-3 records; an empty list is correct for singletons.

**Pipeline** (code lives in `src/er/`, this notebook drives it and shows the evidence behind each design choice):

`raw TSV -> normalisation -> record-centric conjunctive blocking -> stage-1 pair scorer (LightGBM) -> stage-2 S2<->S3 consensus re-scorer (LightGBM) -> precision-biased threshold -> TSV outputs`

**Evidence labels used throughout:** *[measured]* = I ran it on the provided training data (prototype scripts, see section 10 for what is and is not re-measured by this code); *[hypothesis]* = plausible but untested; *[unknown]* = cannot be determined from the data.

**Rules honoured:** only the provided data; no external lookups/APIs/geocoders; the only model family is LightGBM (MIT licence, far below 8B parameters); `text-unidecode` is a local transliteration table, not business data.

## 0. Setup

In [ ]:
import sys, json, time, subprocess
from pathlib import Path
SRC = Path.cwd() / "src"
sys.path.insert(0, str(SRC))
import numpy as np, pandas as pd
from er.config import Params, WORK_DIR, OUT_DIR, find_data_dir
import er.pipeline as pl
from er.data import load_split
from er.textnorm import norm, name_tokens, addr_tokens, skeleton
print("data dir:", find_data_dir()); print("work dir:", WORK_DIR); print("out dir :", OUT_DIR)
params = Params()          # every tunable is in src/er/config.py
params

## 1. Data understanding
Windows note: all multiprocessing lives in `src/er/blocking.py` (importable module), so it works from a notebook.

In [ ]:
s1, pool, gt = load_split("train")
t1, tpool, _ = load_split("test")
print("train: S1 %d | S2+S3 pool %d" % (len(s1), len(pool)))
print("test : S1 %d | S2+S3 pool %d" % (len(t1), len(tpool)))
print(pd.DataFrame({"train_S1": s1.country.value_counts(), "test_S1": t1.country.value_counts(),
                    "train_pool": pool.country.value_counts(), "test_pool": tpool.country.value_counts()}).fillna(0).astype(int))
gt["n"] = gt.matched_entity_ids.map(lambda x: 0 if x == "" else x.count(",") + 1)
print("singleton rate: %.3f | mean matches/S1: %.2f | max %d" % ((gt.n == 0).mean(), gt.n.mean(), gt.n.max()))

### What the exploration established
| Finding | Status | Consequence |
|---|---|---|
| Country label agrees on 100% of 7.6M true pairs (train) | measured | hard partition per country; label kept open-set (France) |
| Every matched S2/S3 id has exactly one owning S1 | measured on train labels | `owner_map` asserts it; only *used* as an optional switch (see 6) |
| ~26.5% of S2/S3 records are orphans; 5.6% of S1 are singletons | measured | orphan/decoy control is the precision lever |
| 39% of S1 names collide with another S1 (different address) | measured | name alone cannot identify |
| India S2/S3: 18% of names in Indic scripts; S1 is 100% ASCII | measured | transliteration + phonetic skeleton needed |
| Orphans look like *decoys*: near-copies of S1 records (link to S1 as often as true records) | measured | most false positives are decoys, not random look-alikes |
| Test pool per S1 is 5.76 vs 4.68 in train | measured fact; **cause unknown** | *not* used to set thresholds |

## 2. Normalisation and blocking keys
Single-token blocking was poor *[measured: <=34% pair recall at K=100 on the full US pool]*. Identity lives in **conjunctions**, so we use hashed token-pair keys:
`A` = address-token pair, `X` = name-token x address-token, `N` = name-token pair.
The v2 key set adds fixes for *measured miss categories*: fused domain/handle names (`prairiefive.com`), house-number corruption (`844` vs `5844`, zero padding), and phonetic-skeleton tokens for transliterated Indic names.

In [ ]:
ex = [("Prairie Five", "389 Greens Road, Houston, TX"), ("#prairiefive", "Texas, 283. Greens Rd, Hoston"),
      ("\u0932\u094b\u091f\u0938 \u090f\u0928\u0930\u094d\u091c\u0940 \u092a\u094d\u0930\u093e\u0907\u0935\u0947\u091f \u0932\u093f\u092e\u093f\u091f\u0947\u0921", "S-D/04 C, THANE, Maharashtra"),
      ("Lotus Energy Private Limited", "S-04 C, 2Nd Floor, Haware Centurian Mall, Thane, Maharashtra")]
for n, a in ex:
    print(repr(n)[:40], "->", name_tokens(n)[:8], "| addr:", addr_tokens(a)[:8])

### Blocking design (record-centric)
* Index the small, deduplicated S1 side once (sorted 64-bit key hashes, memory-mapped so all workers share it).
* Stream every S2/S3 record against it; exact per-key S1 document frequency; keys shared by more than `cap` S1 records are ignored.
* Each record keeps its top-K S1 candidates by key score. `candidate_pairs.tsv` is exactly this set.

**Prototype measurements** (S1-centric prototype on the full US train pool, 3,000 queries, not this exact code path):

| cap | pair recall | avg candidates / S1 |
|---|---|---|
| 20 | 96.3% | 53 |
| 100 | 98.2% | 308 |
| 500 | 99.3% | 1586 |

Candidate counts were light-tailed (US cap 100: p99 = 763, max 1092), so adaptive budgets were not justified there. India was harder (95.0% at 515 candidates; 97.8% at 3077), mostly from non-Latin names and address fragments like "1, Bangalore, KA". The v2 keys lifted US recall at cap 100 from 98.24% to 98.90% (474 candidates); the same cap-50 setting gave 98.48% at 225 candidates, a better trade-off than v1 cap 100. India v2 was not completed.

## 3. Pair features
`src/er/features.py`, vectorised with `rapidfuzz.process.cpdist` (about 1 microsecond/pair/scorer).
* Name: ratio, token-set, partial, token-sort on normalised names; **phonetic-skeleton** ratio (measured AUC 0.994 vs 0.970 for plain transliteration on non-Latin India names vs *random* names); length; non-Latin and domain/handle flags.
* Address: token-set/ratio/partial, number Jaccard, **number-set equality** (decoys differ by a house number), empty flag, length.
* Retrieval evidence: per-type key scores/counts, best-key rarity, share and gap versus the record's best S1, number of S1 candidates.
* **Excluded on purpose:** absolute S1-name-frequency. It helped India (+0.009) but test S1 is a differently sized sample (US dup-name rate 0.29 vs 0.36), a measured shift risk.

Feature importance and ablations *[measured, prototype, US / India, grouped 3-fold F0.5 incl. singletons]*:

| Features | US | India |
|---|---|---|
| name only | 0.639 | 0.625 |
| address only | 0.732 | 0.701 |
| name + address | 0.928 | 0.895 |
| + retrieval keys | 0.941 | 0.908 |

Neither field dominates: address is somewhat stronger, but the two are strongly complementary.

## 4. Stage 2: S2<->S3 consensus
False positives are mostly near-duplicate **decoys** (same name, perturbed house number or suffix). A true member agrees with *other* confident members (independent noise); a decoy does not. Stage 2 adds those relational features (`src/er/relational.py`).

Prototype measurement: F0.5 US 0.942 -> 0.949, India 0.916 -> 0.935. Listwise-only features gave about 0.

## 5. Train + validate (accuracy-gated)
`pl.train` builds cluster-closed universes (sampled S1 + *all* their true records + a proportional share of orphans), runs the real blocking, produces **out-of-fold** predictions grouped by S1 name (so same-name chains never straddle folds), selects the threshold, and reports:
candidate recall, the **ceiling** (F0.5 with perfect scoring inside the candidate set), F0.5 per stage, per country, singleton vs non-singleton, precision/recall.

**Accuracy gate:** the pipeline refuses to produce submission files unless every country's grouped-OOF macro F0.5 is at least `Params.min_val_f05` (0.90).
Caveat: sampled universes dilute same-name confounders relative to full scale, so real-scale F0.5 can be somewhat lower.

Increase `n_s1_per_country` for a stronger fit (RAM permitting).

In [ ]:
N_S1 = {"US": 150_000, "India": 150_000}      # training universe size per country
t0 = time.time()
report, D, S = pl.train(params, N_S1)
(WORK_DIR / "validation_report.json").write_text(json.dumps(report, indent=1))
print("train+validate took %.1f min" % ((time.time() - t0) / 60))
print(json.dumps({k: v for k, v in report.items() if "by_threshold" not in k}, indent=1))

In [ ]:
import matplotlib.pyplot as plt
s1s, s2s = report["stage1_F05_by_threshold"], report["stage2_F05_by_threshold"]
plt.figure(figsize=(6, 3.5))
plt.plot(list(s1s), list(s1s.values()), "o-", label="stage 1")
plt.plot(list(s2s), list(s2s.values()), "o-", label="stage 2 (+consensus)")
plt.axvline(report["threshold"], ls="--", c="gray"); plt.axhline(params.min_val_f05, ls=":", c="red")
plt.xlabel("probability threshold"); plt.ylabel("grouped OOF macro F0.5"); plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
assert report["passes_gate"], "Validation is below the accuracy gate - do not submit; improve the model first."

## 6. Error analysis on the out-of-fold predictions
Decomposes the loss into the four separate sources: **blocking miss** (1 - ceiling), **pair scoring**, **global decision**, and the metric.

In [ ]:
th = report["threshold"]
print("candidate pair recall : %.4f" % report["candidate_recall_pairs"])
print("ceiling F0.5          : %.4f   -> blocking loss = %.4f" % (report["ceiling_F05"], 1 - report["ceiling_F05"]))
print("stage-1 best F0.5     : %.4f" % max(report["stage1_F05_by_threshold"].values()))
print("stage-2 F0.5 @ th=%.2f : %.4f   -> scoring loss = %.4f" % (th, report["stage2_F05_at_threshold"], report["ceiling_F05"] - report["stage2_F05_at_threshold"]))
print("micro precision %.4f | micro recall %.4f" % (report["micro_precision"], report["micro_recall_vs_all_true"]))
for c, v in report["per_country"].items(): print(c, {k: round(x, 4) for k, x in v.items()})
sel = S[S.p2 >= th]
fp = sel[sel.y == 0]; fn = S[(S.y == 1) & (S.p2 < th)]
print("false positives: %d | false negatives inside candidate set: %d" % (len(fp), len(fn)))

Prototype error analysis *[measured]*: about 70-75% of false positives were orphans with no owner (so the one-owner constraint cannot remove them); they are near-duplicate decoys. Missed true pairs were dominated by generic names with empty/degraded addresses and by house-number corruption; **true pairs also carry number typos**, so some ambiguity is irreducible from these fields.

**One-owner assignment** (each S2/S3 record keeps only its best S1): measured gain about +0.001 at best, so `Params.one_owner=False`.
**Foundation models:** not used. The phonetic skeleton already separates non-Latin names (AUC 0.994 vs random); a multilingual encoder is *unproven* to help and would add download/licence/compute burden. *[hypothesis]* it could help hard negatives; not tested.

## 7. Inference on the test set
`pl.infer` processes every country label present (open set, so France is handled by the same generic code), chunk by chunk to bound RAM, and refuses to run if the accuracy gate failed.

France has **no labels**, so its accuracy cannot be measured here. What *is* known from the provided test files: shorter names (3.1 tokens), 15.6% of S1 names carry diacritics, and 11% of S1 records share an address with another S1 (vs 4-6% elsewhere), so address is less decisive. Check the printed France diagnostics (singleton rate, matches per S1) against US/India for sanity.

In [ ]:
t0 = time.time()
ids, matches, cands, diag = pl.infer(params)
pl.write_outputs(ids, matches, cands)
(WORK_DIR / "test_diagnostics.json").write_text(json.dumps(diag, indent=1))
print("inference took %.1f min" % ((time.time() - t0) / 60))
pd.DataFrame(diag).T

## 8. Validate the submission format with the official script

In [ ]:
dd = find_data_dir()
val = dd.parent / "utils" / "validate_submission.py"
r = subprocess.run([sys.executable, str(val), "--matching", str(OUT_DIR / "matching_results.tsv"),
                    "--candidate", str(OUT_DIR / "candidate_pairs.tsv"), "--test-dir", str(dd / "test")],
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-1000:])

## 9. What we do and do not know
* **[unknown]** Test orphan prevalence. The test pool is larger per S1 than train, but one story (extra orphans) or another (owners missing from test S1) fit equally; a linkage statistic could not separate them because orphans are decoys that link like true records. The threshold is therefore chosen at the *upper end of the flat F0.5 plateau* (precision insurance), not tuned to a guessed rate.
* **[unknown]** France accuracy (no labels). Nothing here has been validated on French data.
* **[hypothesis]** Real-scale F0.5 is a bit below the sampled-universe number because same-name confounders are diluted in small universes.
* Transductive use of test data (e.g. mining aliases from confident test pairs) is **not** used: not needed for the train-only baseline, and the rules on it are not explicit. Revisit only after the train-only baseline is fixed.

## 10. Reproducibility and what is measured where
All numbers in sections 1-4 and 6 marked *[measured]* came from prototype scripts on the training data (grouped folds, US/India, 3,000 in-sample queries); this package re-computes the headline metrics in section 5 with its own code, so treat section 5's printed report as the authoritative validation of *this* implementation.